# Install Necessary Libs

In [6]:
pip install requests beautifulsoup4

# Import Libs

In [7]:
"""
Job scraper for https://realpython.github.io/fake-jobs/
Extracts job title, company, location, and detail URL for each listing.
"""

import csv
import requests
from bs4 import BeautifulSoup

# Base URL & Output File Defination

In [8]:
BASE_URL = "https://realpython.github.io/fake-jobs/"
OUTPUT_FILE = "jobs.csv"

# Fetch Data from Base URL

In [9]:
def fetch_page(url: str) -> str:
    """Fetch the HTML content of a page."""
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        return response.text
    except requests.RequestException as e:
        raise RuntimeError(f"Failed to fetch {url}: {e}")

# Parse Jobs

In [10]:
def parse_jobs(html: str) -> list[dict]:
    """Parse job listings from the HTML."""
    soup = BeautifulSoup(html, "html.parser")
    jobs = []

    # Each job card has class "card-content"
    for card in soup.find_all("div", class_="card-content"):
        # The <h2> holds title, <h3> holds company, <p> holds location
        title_tag = card.find("h2", class_="title")
        company_tag = card.find("h3", class_="company")
        location_tag = card.find("p", class_="location")

        # The detail link is the anchor inside the footer of the card
        footer = card.find("footer", class_="card-footer")
        link_tag = footer.find("a") if footer else None

        # Build a dict with safe fallbacks for missing fields
        job = {
            "title": title_tag.get_text(strip=True) if title_tag else "",
            "company": company_tag.get_text(strip=True) if company_tag else "",
            "location": location_tag.get_text(strip=True) if location_tag else "",
            "url": link_tag["href"] if link_tag and link_tag.has_attr("href") else "",
        }
        jobs.append(job)

    return jobs

# Save Data to CSV File

In [11]:
def save_to_csv(jobs: list[dict], filename: str) -> None:
    """Write job listings to a CSV file."""
    if not jobs:
        print("No jobs to save.")
        return

    fieldnames = ["title", "company", "location", "url"]
    with open(filename, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(jobs)

# Run Application

In [12]:
def main() -> None:
    print(f"Fetching {BASE_URL} ...")
    html = fetch_page(BASE_URL)

    jobs = parse_jobs(html)
    print(f"Found {len(jobs)} job listings.")

    save_to_csv(jobs, OUTPUT_FILE)
    print(f"Saved to {OUTPUT_FILE}")


if __name__ == "__main__":
    main()

Fetching https://realpython.github.io/fake-jobs/ ...
Found 100 job listings.
Saved to jobs.csv


# Check file CSV

In [13]:
# Xem 5 dòng đầu
import pandas as pd
df = pd.read_csv("jobs.csv")
print(df.head())
print(f"\nTổng số dòng: {len(df)}")
print(f"Các cột: {list(df.columns)}")

                     title                     company              location  \
0  Senior Python Developer    Payne, Roberts and Davis       Stewartbury, AA   
1          Energy engineer            Vasquez-Davidson  Christopherville, AA   
2          Legal executive  Jackson, Chambers and Levy   Port Ericaburgh, AA   
3   Fitness centre manager              Savage-Bradley     East Seanview, AP   
4          Product manager                 Ramirez Inc   North Jamieview, AP   

                          url  
0  https://www.realpython.com  
1  https://www.realpython.com  
2  https://www.realpython.com  
3  https://www.realpython.com  
4  https://www.realpython.com  

Tổng số dòng: 100
Các cột: ['title', 'company', 'location', 'url']


# Download file CSV into local or Driver

In [16]:
# from google.colab import files
# files.download("jobs.csv")

from google.colab import drive
drive.mount('/content/drive')

# Ghi thẳng vào Drive
import shutil
shutil.copy("jobs.csv", "/content/drive/MyDrive/jobs.csv")

Mounted at /content/drive


'/content/drive/MyDrive/jobs.csv'

# Check quality of data

In [17]:
# Đếm số field bị thiếu
for col in df.columns:
    missing = df[col].isna().sum() + (df[col].astype(str).str.strip() == "").sum()
    print(f"{col:10s}: {missing} giá trị thiếu")

# Kiểm tra tất cả URL có hợp lệ không
print("\nTất cả URL bắt đầu bằng https://realpython.github.io/fake-jobs/jobs/:",
      df["url"].str.startswith("https://realpython.github.io/fake-jobs/jobs/").all())

title     : 0 giá trị thiếu
company   : 0 giá trị thiếu
location  : 0 giá trị thiếu
url       : 0 giá trị thiếu

Tất cả URL bắt đầu bằng https://realpython.github.io/fake-jobs/jobs/: False
